In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
ds = pd.read_csv("ckd-dataset-v2.csv")
ds = ds.drop([0, 1]).reset_index(drop=True)

target_col = 'age'

In [3]:
def clean_range(value):
    if pd.isna(value): return np.nan
    value = str(value).strip()
    if '<' in value or '>' in value or '≥' in value or '≤' in value:
        clean_num = value.replace('<', '').replace('>', '').replace('≥', '').replace('≤', '').replace('=', '').strip()
        try: return float(clean_num)
        except ValueError: return np.nan
    if '-' in value:
        parts = value.split('-')
        try: return (float(parts[0].strip()) + float(parts[1].strip())) / 2.0
        except ValueError: return np.nan
    try: return float(value)
    except ValueError: return np.nan

binned_cols = ['sg', 'al', 'su', 'bu', 'bgr', 'sod', 'sc', 'pot', 'hemo', 'pcv', 'rbcc', 'wbcc', 'grf', 'age']
for col in binned_cols:
    if col in ds.columns:
        ds[col] = ds[col].apply(clean_range)

numeric_suspects = ['bp (Diastolic)', 'bp limit', 'rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']
for col in numeric_suspects:
    if col in ds.columns:
        ds[col] = pd.to_numeric(ds[col], errors='coerce')

In [4]:
y = ds[target_col].values
X = ds.drop(columns=[target_col])

In [5]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object', 'category']).columns

print(f"Detected Numeric Features: {list(numeric_features)}")
print(f"Detected Categorical Features: {list(categorical_features)}")

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

X_clean = preprocessor.fit_transform(X)

Detected Numeric Features: ['bp (Diastolic)', 'bp limit', 'sg', 'al', 'rbc', 'su', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sod', 'sc', 'pot', 'hemo', 'pcv', 'rbcc', 'wbcc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'grf']
Detected Categorical Features: ['class', 'stage', 'affected']


In [6]:
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
y_clean = imputer.fit_transform(y.reshape(-1, 1)).flatten()

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_clean, y_clean, test_size = 0.2, random_state = 0)

In [10]:
from sklearn.linear_model import LinearRegression
regressor = LinearRegression()
regressor.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [12]:
y_pred = regressor.predict(X_test)
print(y_pred)

[37.88044747 61.60307248 61.4462214  60.78961391 42.40212562 64.05235958
 48.75734782 63.84507804 52.46394685 66.3419502  56.48294031 45.99773119
 56.50044514 52.46357008 43.9234182  47.48693105 45.52802747 48.53619555
 43.84128357 62.70033243 58.46302401 63.52457402 42.49512783 57.34555036
 38.51833379 53.62618637 57.92771692 51.91762762 59.00445897 36.51689721
 31.24873884 47.90377277 64.59387129 46.55519806 52.25812561 67.24237982
 42.37364238 46.2707079  65.15122603 54.23034829]


In [13]:
from sklearn.metrics import r2_score
r2_score(y_test, y_pred)

0.23426932612822515